In [0]:
# ============================================================
# WEATHER DATA ENGINEERING PIPELINE
# Open-Meteo API
# PySpark
# Delta Lake
# Bronze → Silver → Gold
# + Data Quality Checks
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import requests

from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    max,
    min,
    round,
    to_date,
    to_timestamp,
    when,
)

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

LATITUDE = 41.0225
LONGITUDE = 28.9408

BRONZE_TABLE = "default.weather_bronze"
SILVER_TABLE = "default.weather_silver"
GOLD_TABLE = "default.weather_gold"


# ============================================================
# 3. EXTRACT DATA FROM OPEN-METEO API
# ============================================================

url = (
    "https://api.open-meteo.com/v1/forecast"
    f"?latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    "&current_weather=true"
)

response = requests.get(
    url,
    timeout=30
)


# Check API response
if response.status_code != 200:
    raise Exception(
        f"Weather API request failed: "
        f"{response.status_code}"
    )


# Parse JSON
data = response.json()["current_weather"]


print("========================================")
print("Weather data fetched successfully")
print("========================================")

print(data)


# ============================================================
# 4. CREATE SPARK DATAFRAME
# ============================================================

schema = StructType([

    StructField(
        "latitude",
        DoubleType(),
        True
    ),

    StructField(
        "longitude",
        DoubleType(),
        True
    ),

    StructField(
        "temperature",
        DoubleType(),
        True
    ),

    StructField(
        "windspeed",
        DoubleType(),
        True
    ),

    StructField(
        "winddirection",
        DoubleType(),
        True
    ),

    StructField(
        "weathercode",
        IntegerType(),
        True
    ),

    StructField(
        "weather_time",
        StringType(),
        True
    ),
])


row_data = [(
    LATITUDE,
    LONGITUDE,
    float(data["temperature"]),
    float(data["windspeed"]),
    float(data["winddirection"]),
    int(data["weathercode"]),
    data["time"],
)]


df = spark.createDataFrame(
    row_data,
    schema
)


# ============================================================
# 5. BRONZE DATAFRAME
# Raw API data
# ============================================================

df_bronze = (
    df
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)


print("\n========================================")
print("BRONZE DATA")
print("========================================")

display(df_bronze)


# ============================================================
# 6. DATA QUALITY CHECKS
# ============================================================

print("\n========================================")
print("RUNNING DATA QUALITY CHECKS")
print("========================================")


# ------------------------------------------------------------
# Check 1: DataFrame should not be empty
# ------------------------------------------------------------

if df_bronze.isEmpty():

    raise Exception(
        "Data Quality Failed: "
        "DataFrame is empty"
    )

print("✓ Check 1 passed: DataFrame is not empty")


# ------------------------------------------------------------
# Check 2: Temperature should be between -50 and 60
# ------------------------------------------------------------

invalid_temperature = (
    df_bronze
    .filter(
        ~col("temperature").between(-50, 60)
    )
    .count()
)


if invalid_temperature > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{invalid_temperature} invalid temperature records"
    )

print(
    "✓ Check 2 passed: "
    "Temperature values are valid"
)


# ------------------------------------------------------------
# Check 3: Wind speed should not be negative
# ------------------------------------------------------------

invalid_wind = (
    df_bronze
    .filter(
        col("windspeed") < 0
    )
    .count()
)


if invalid_wind > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{invalid_wind} invalid windspeed records"
    )

print(
    "✓ Check 3 passed: "
    "Windspeed values are valid"
)


# ------------------------------------------------------------
# Check 4: Latitude should not be NULL
# ------------------------------------------------------------

null_latitude = (
    df_bronze
    .filter(
        col("latitude").isNull()
    )
    .count()
)


if null_latitude > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{null_latitude} NULL latitude records"
    )

print(
    "✓ Check 4 passed: "
    "Latitude is not NULL"
)


# ------------------------------------------------------------
# Check 5: Longitude should not be NULL
# ------------------------------------------------------------

null_longitude = (
    df_bronze
    .filter(
        col("longitude").isNull()
    )
    .count()
)


if null_longitude > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{null_longitude} NULL longitude records"
    )

print(
    "✓ Check 5 passed: "
    "Longitude is not NULL"
)


# ------------------------------------------------------------
# Check 6: Weather code should not be NULL
# ------------------------------------------------------------

null_weathercode = (
    df_bronze
    .filter(
        col("weathercode").isNull()
    )
    .count()
)


if null_weathercode > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{null_weathercode} NULL weathercode records"
    )

print(
    "✓ Check 6 passed: "
    "Weather code is not NULL"
)


# ------------------------------------------------------------
# Check 7: Weather time should not be NULL
# ------------------------------------------------------------

null_weather_time = (
    df_bronze
    .filter(
        col("weather_time").isNull()
    )
    .count()
)


if null_weather_time > 0:

    raise Exception(
        f"Data Quality Failed: "
        f"{null_weather_time} NULL weather_time records"
    )

print(
    "✓ Check 7 passed: "
    "Weather time is not NULL"
)


print("\n========================================")
print("ALL DATA QUALITY CHECKS PASSED")
print("========================================")


# ============================================================
# 7. WRITE BRONZE LAYER
# ============================================================

# During development:
# Remove old table to avoid schema mismatch

spark.sql(
    f"DROP TABLE IF EXISTS {BRONZE_TABLE}"
)


(
    df_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)


print(
    "\n✓ Bronze layer written successfully"
)


# ============================================================
# 8. SILVER LAYER
# Cleaning + Transformation
# ============================================================

df_silver = (

    df_bronze

    # --------------------------------------------------------
    # Temperature validation
    # --------------------------------------------------------

    .filter(
        col("temperature").between(-50, 60)
    )

    # --------------------------------------------------------
    # Wind speed validation
    # --------------------------------------------------------

    .filter(
        col("windspeed") >= 0
    )

    # --------------------------------------------------------
    # Latitude validation
    # --------------------------------------------------------

    .filter(
        col("latitude").isNotNull()
    )

    # --------------------------------------------------------
    # Longitude validation
    # --------------------------------------------------------

    .filter(
        col("longitude").isNotNull()
    )

    # --------------------------------------------------------
    # Weather description
    # --------------------------------------------------------

    .withColumn(

        "weather_description",

        when(
            col("weathercode") == 0,
            "Clear sky"
        )

        .when(
            col("weathercode").isin(1, 2, 3),
            "Cloudy"
        )

        .when(
            col("weathercode").isin(45, 48),
            "Fog"
        )

        .when(
            col("weathercode").isin(51, 53, 55),
            "Drizzle"
        )

        .when(
            col("weathercode").isin(61, 63, 65),
            "Rain"
        )

        .when(
            col("weathercode").isin(71, 73, 75),
            "Snow"
        )

        .otherwise(
            "Other"
        )
    )

    # --------------------------------------------------------
    # Rename temperature
    # --------------------------------------------------------

    .withColumnRenamed(
        "temperature",
        "temperature_celsius"
    )

    # --------------------------------------------------------
    # Convert weather_time to timestamp
    # --------------------------------------------------------

    .withColumn(
        "weather_time",
        to_timestamp("weather_time")
    )
)


print("\n========================================")
print("SILVER DATA")
print("========================================")

display(df_silver)


# ============================================================
# 9. WRITE SILVER LAYER
# ============================================================

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(SILVER_TABLE)
)


print(
    "✓ Silver layer written successfully"
)


# ============================================================
# 10. GOLD LAYER
# Analytics-ready data
# ============================================================

df_gold = (

    df_silver

    # --------------------------------------------------------
    # Extract date from timestamp
    # --------------------------------------------------------

    .withColumn(
        "weather_date",
        to_date("weather_time")
    )

    # --------------------------------------------------------
    # Group by date and location
    # --------------------------------------------------------

    .groupBy(
        "weather_date",
        "latitude",
        "longitude"
    )

    # --------------------------------------------------------
    # Aggregations
    # --------------------------------------------------------

    .agg(

        # Average temperature
        round(
            avg("temperature_celsius"),
            2
        ).alias(
            "avg_temperature_celsius"
        ),

        # Minimum temperature
        round(
            min("temperature_celsius"),
            2
        ).alias(
            "min_temperature_celsius"
        ),

        # Maximum temperature
        round(
            max("temperature_celsius"),
            2
        ).alias(
            "max_temperature_celsius"
        ),

        # Average wind speed
        round(
            avg("windspeed"),
            2
        ).alias(
            "avg_windspeed"
        ),

        # Number of readings
        count("*").alias(
            "reading_count"
        )
    )
)


print("\n========================================")
print("GOLD DATA")
print("========================================")

display(df_gold)


# ============================================================
# 11. WRITE GOLD LAYER
# ============================================================

(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(GOLD_TABLE)
)


print(
    "✓ Gold layer written successfully"
)


# ============================================================
# 12. FINAL DATA QUALITY CHECK
# ============================================================

print("\n========================================")
print("FINAL GOLD DATA QUALITY CHECK")
print("========================================")


# Gold should not be empty

if df_gold.isEmpty():

    raise Exception(
        "Final Data Quality Failed: "
        "Gold table is empty"
    )


# Check average temperature

invalid_gold_temperature = (
    df_gold
    .filter(
        ~col("avg_temperature_celsius")
        .between(-50, 60)
    )
    .count()
)


if invalid_gold_temperature > 0:

    raise Exception(
        "Final Data Quality Failed: "
        "Invalid Gold temperature"
    )


print(
    "✓ Gold data quality check passed"
)


# ============================================================
# 13. FINAL RESULT
# ============================================================

print("\n========================================")
print("WEATHER PIPELINE COMPLETED SUCCESSFULLY")
print("========================================")


print("\nBRONZE TABLE:")
display(
    spark.sql(
        f"SELECT * FROM {BRONZE_TABLE}"
    )
)


print("\nSILVER TABLE:")
display(
    spark.sql(
        f"SELECT * FROM {SILVER_TABLE}"
    )
)


print("\nGOLD TABLE:")
display(
    spark.sql(
        f"SELECT * FROM {GOLD_TABLE}"
    )
)

Weather data fetched successfully
{'time': '2026-09-08T19:00', 'interval': 900, 'temperature': 21.1, 'windspeed': 13.0, 'winddirection': 46, 'is_day': 0, 'weathercode': 0}

BRONZE DATA


latitude,longitude,temperature,windspeed,winddirection,weathercode,weather_time,ingested_at
41.0225,28.9408,21.1,13.0,46.0,0,2026-09-08T19:00,2026-09-08T19:14:49.065Z



RUNNING DATA QUALITY CHECKS
✓ Check 1 passed: DataFrame is not empty
✓ Check 2 passed: Temperature values are valid
✓ Check 3 passed: Windspeed values are valid
✓ Check 4 passed: Latitude is not NULL
✓ Check 5 passed: Longitude is not NULL
✓ Check 6 passed: Weather code is not NULL
✓ Check 7 passed: Weather time is not NULL

ALL DATA QUALITY CHECKS PASSED

✓ Bronze layer written successfully

SILVER DATA


latitude,longitude,temperature_celsius,windspeed,winddirection,weathercode,weather_time,ingested_at,weather_description
41.0225,28.9408,21.1,13.0,46.0,0,2026-09-08T19:00:00.000Z,2026-09-08T19:14:55.539Z,Clear sky


✓ Silver layer written successfully

GOLD DATA


...

latitude,longitude,temperature,windspeed,winddirection,weathercode,weather_time,ingested_at
41.0225,28.9408,21.1,13.0,46.0,0,2026-09-08T19:00,2026-09-08T19:14:54.170Z



SILVER TABLE:


latitude,longitude,temperature_celsius,windspeed,winddirection,weathercode,weather_time,ingested_at,weather_description
41.0225,28.9408,21.1,13.0,46.0,0,2026-09-08T19:00:00.000Z,2026-09-08T19:14:56.881Z,Clear sky



GOLD TABLE:


weather_date,latitude,longitude,avg_temperature_celsius,min_temperature_celsius,max_temperature_celsius,avg_windspeed,reading_count
2026-09-08,41.0225,28.9408,21.1,21.1,21.1,13.0,1
